# Deploy AgentCore Agent with Long-Term Memory

This notebook deploys a memory-enabled agent from Module 6 infrastructure.

**Prerequisites:** Module 6 must be completed first (Gateway + Lambdas deployed).

## What You'll Do

1. Create an AgentCore Memory resource
2. Deploy a second agent with `memory_mode="STM_AND_LTM"`
3. Test cross-session memory recall (same actor, different sessions)

In [ ]:
import boto3
import json
import time
import os
import glob
import uuid

REGION = os.environ.get("AWS_REGION", "us-east-1")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# AWS clients
iam = boto3.client("iam")
agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)
codebuild_client = boto3.client("codebuild", region_name=REGION)

# Resource names from Module 6
BOOKINGS_TABLE = "workshop-Bookings"
AGENTCORE_ROLE_NAME = "workshop-AgentCoreExecutionRole"
GATEWAY_NAME = "HotelBookingGateway"

# --- Workshop resource tag ---
# 08-cleanup/workshop_cleanup.py deletes a resource ONLY if it carries this exact
# key and value. A near-miss value is the same as no tag at all: cleanup reports
# UNTAGGED_BLOCKED, exits non-zero, and leaves billable resources running.
# The four shapes below are not interchangeable. Each service demands its own.
WORKSHOP_TAG_KEY = "WorkshopResource"
WORKSHOP_TAG_VALUE = "stop-ai-agent-hallucinations"
WORKSHOP_TAGS_MAP = {WORKSHOP_TAG_KEY: WORKSHOP_TAG_VALUE}                        # lambda, agentcore
WORKSHOP_TAGS_KV = [{"Key": WORKSHOP_TAG_KEY, "Value": WORKSHOP_TAG_VALUE}]       # dynamodb, iam, ecr
WORKSHOP_TAGS_KV_LOWER = [{"key": WORKSHOP_TAG_KEY, "value": WORKSHOP_TAG_VALUE}] # codebuild

# Recover variables from Module 6 deployment
try:
    AGENTCORE_ROLE_ARN = iam.get_role(RoleName=AGENTCORE_ROLE_NAME)["Role"]["Arn"]
    print(f"AgentCore role: {AGENTCORE_ROLE_ARN}")
except Exception as e:
    raise RuntimeError(f"Module 6 role not found. Deploy Module 6 first. Error: {e}")

try:
    gateways = agentcore.list_gateways()["items"]
    gw = next((g for g in gateways if g["name"] == GATEWAY_NAME), None)
    if not gw:
        raise RuntimeError("Gateway not found")
    GATEWAY_ID = gw["gatewayId"]
    gw_info = agentcore.get_gateway(gatewayIdentifier=GATEWAY_ID)
    GATEWAY_URL = gw_info["gatewayUrl"]
    print(f"Gateway: {GATEWAY_ID}")
    print(f"Gateway URL: {GATEWAY_URL}")
except Exception as e:
    raise RuntimeError(f"Module 6 Gateway not found. Deploy Module 6 first. Error: {e}")

print("\n✓ Module 6 resources recovered. Ready to deploy memory-enabled agent.")

---
## Architecture: Two Agents, Same Infrastructure

This module deploys a **second agent** with long-term memory alongside the Module 6 agent:

| Agent | Module | Memory Mode | Behavior |
|-------|--------|-------------|----------|
| **HotelBookingAgent** | Module 6 | Runtime memory only | Ephemeral conversation buffer - no persistence |
| **HotelBookingAgentWithMemory** | Module 7 | `STM_AND_LTM` | Extracts strategies, recalls across sessions |

**Both agents reuse the SAME infrastructure:**
- AgentCore Gateway (`HotelBookingGateway`)
- Lambda functions (8 booking + graph tools)
- DynamoDB tables (Hotels, Bookings, SteeringRules)
- IAM roles

**What Module 7 adds:**

1. **AgentCore Memory resource** with strategies:
   - `UserPreferences` — hotel star rating, preferred cities
   - `UserFacts` — user name, loyalty numbers
   
2. **New agent code** (`booking_agent_with_memory.py`):
   - Imports `AgentCoreMemoryConfig` and `AgentCoreMemorySessionManager`
   - Extracts `actor_id` from custom HTTP header (`X-Amzn-Bedrock-AgentCore-Runtime-Custom-Actor-Id`)
   - Configures memory retrieval paths for user facts and preferences
   
3. **Deploy with `memory_mode="STM_AND_LTM"`** — enables both short-term (within session) and long-term (across sessions) memory

4. **Pass `BEDROCK_AGENTCORE_MEMORY_ID`** as environment variable to link the agent to the Memory resource

**Memory types explained:**

- **Runtime memory** (Module 6): Temporary conversation buffer maintained by the Strands Agent runtime. Lost when session ends.
- **STM (Short-Term Memory)**: Session-scoped memory managed by AgentCore. Lost when session ends.
- **LTM (Long-Term Memory)**: Persistent memory managed by AgentCore. Extracts strategies asynchronously and recalls across sessions.

**Code comparison:**

```python
# booking_agent.py (Module 6 — runtime memory only)
@app.entrypoint
def invoke(payload, context=None):
    agent = Agent(model=model, tools=tools, system_prompt=SYSTEM_PROMPT, hooks=hooks)
    result = agent(prompt)
    return str(result)
```

```python
# booking_agent_with_memory.py (Module 7 — with AgentCore Memory)
@app.entrypoint
def invoke(payload, context: RequestContext = None):
    # Extract actor ID from custom header
    actor_id = context.request_headers.get('x-amzn-bedrock-agentcore-runtime-custom-actor-id', 'default-user')
    
    # Configure AgentCore Memory
    memory_config = AgentCoreMemoryConfig(
        memory_id=MEMORY_ID,
        session_id=context.session_id,
        actor_id=actor_id,
        retrieval_config={
            f"/users/{actor_id}/facts": RetrievalConfig(top_k=3, relevance_score=0.5),
            f"/users/{actor_id}/preferences": RetrievalConfig(top_k=3, relevance_score=0.5)
        }
    )
    session_manager = AgentCoreMemorySessionManager(memory_config, region)
    
    agent = Agent(model=model, tools=tools, system_prompt=SYSTEM_PROMPT, 
                  hooks=hooks, session_manager=session_manager)
    result = agent(prompt)
    return {"response": result.message.get("content", [{}])[0].get("text", str(result))}
```

**Key difference:** Module 7 integrates AgentCore Memory via `AgentCoreMemorySessionManager`, enabling persistent memory across sessions.

---
## Step 1: Create AgentCore Memory Resource

Before deploying the memory-enabled agent, we need to create an AgentCore Memory resource with two strategies:

1. **UserPreferences** — stores hotel preferences (star rating, cities, etc.)
2. **UserFacts** — stores factual information (name, loyalty number, etc.)

These strategies define what the agent should extract and remember across sessions.

In [ ]:
# Create AgentCore Memory resource
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_control = boto3.client('bedrock-agentcore-control', region_name=REGION)

MEMORY_NAME = "workshop_HotelBookingMemory"


def memory_id_matches_name(memory_id: str, memory_name: str) -> bool:
    """AgentCore mints memory ids as ``<name>-<suffix>``.

    ``list_memories`` returns no ``name`` field at all. A MemorySummary is
    ``arn, id, status, createdAt, updatedAt, managedByResourceArn``, which is why
    the original code reached for ``memories['memories'][0]`` and tagged whatever
    happened to sort first. Cleanup then deletes anything carrying the workshop
    tag, so that could adopt and then destroy an unrelated memory. See bug B46.

    This strips exactly one trailing ``-<suffix>`` and requires the remainder to
    EQUAL the name. It is an equality test, not a prefix test:
    ``workshop_HotelBookingMemoryExtra-abc`` does not match. It is the same rule
    ``08-cleanup/workshop_cleanup.py`` uses to decide what to delete, so what
    Module 7 adopts is exactly what Module 8 will remove.
    """
    if memory_id == memory_name:
        return True
    head, sep, _suffix = memory_id.rpartition("-")
    return bool(sep) and head == memory_name


def find_workshop_memory():
    """Return this workshop's memory, or None. Never selects by index position.

    Zero matches -> None, and the caller creates the memory. That is the normal
    first-run path. More than one match -> stop, because guessing is exactly the
    failure this function exists to prevent.
    """
    matches = [
        memory
        for page in agentcore_control.get_paginator("list_memories").paginate()
        for memory in page.get("memories", [])
        if memory_id_matches_name(memory["id"], MEMORY_NAME)
    ]
    if len(matches) > 1:
        raise RuntimeError(
            f"{len(matches)} memories match the name {MEMORY_NAME!r}: "
            f"{[m['id'] for m in matches]}. Refusing to guess which one is the "
            "workshop's. Delete the extras by hand, then re-run this cell."
        )
    return matches[0] if matches else None


def adopt_workshop_memory():
    """Tag and reuse the existing workshop memory. Raises if it cannot be found."""
    existing = find_workshop_memory()
    if existing is None:
        raise RuntimeError(
            f"AgentCore reports {MEMORY_NAME!r} already exists, but no memory in "
            f"{REGION} carries that name. Refusing to tag an unrelated memory. "
            "Check the region, or delete the conflicting memory by hand."
        )
    agentcore_control.tag_resource(resourceArn=existing["arn"], tags=WORKSHOP_TAGS_MAP)
    print(f"Using existing memory: {existing['id']} - tagged")
    return existing["id"]


print("Creating AgentCore Memory resource...")

try:
    memory_response = agentcore_control.create_memory(
        name=MEMORY_NAME,
        description="Long-term memory for hotel booking agent - stores user preferences",
        eventExpiryDuration=30,  # Keep events for 30 days
        memoryStrategies=[
            {
                'userPreferenceMemoryStrategy': {
                    'name': 'UserPreferences',
                    'description': 'Store user hotel preferences (stars, cities, etc.)',
                    'namespaces': ['/users/{actorId}/preferences']
                }
            },
            {
                'semanticMemoryStrategy': {
                    'name': 'UserFacts',
                    'description': 'Store factual information about users',
                    'namespaces': ['/users/{actorId}/facts']
                }
            }
        ],
        tags=WORKSHOP_TAGS_MAP,  # AgentCore: lowercase tags=, map shape
    )
    MEMORY_ID = memory_response['memory']['id']
    print(f"Created memory: {MEMORY_ID}")

except agentcore_control.exceptions.ConflictException:
    print(f"Memory '{MEMORY_NAME}' already exists")
    MEMORY_ID = adopt_workshop_memory()

except agentcore_control.exceptions.ValidationException as e:
    if 'already exists' not in str(e):
        raise
    print(f"Memory '{MEMORY_NAME}' already exists")
    MEMORY_ID = adopt_workshop_memory()

# Wait for memory to be ACTIVE
print("Waiting for memory to be ACTIVE...")
for _ in range(30):
    memory_status = agentcore_control.get_memory(memoryId=MEMORY_ID)
    if memory_status['memory']['status'] == 'ACTIVE':
        break
    time.sleep(5)

print(f"Memory status: {memory_status['memory']['status']}")
print(f"Memory ID: {MEMORY_ID}")

# Verify rather than trust: cleanup deletes memory only if this tag is present.
memory_arn = memory_status['memory']['arn']
memory_tags = agentcore_control.list_tags_for_resource(resourceArn=memory_arn).get('tags', {})
if memory_tags.get(WORKSHOP_TAG_KEY) != WORKSHOP_TAG_VALUE:
    raise RuntimeError(f"Memory tag did not stick. Read back: {memory_tags}")
print(f"Memory tagged: {WORKSHOP_TAG_KEY}={WORKSHOP_TAG_VALUE}")


---
## Step 2: Deploy Memory-Enabled Agent

Now deploy a second agent configured with `memory_mode="STM_AND_LTM"`. This agent will:

- Use the Memory resource created above (`BEDROCK_AGENTCORE_MEMORY_ID` env var)
- Extract strategies from conversations asynchronously
- Recall strategies across different sessions (same actor ID)

**Deployment time:** 3-5 minutes (builds container + pushes to ECR)

In [ ]:
# Pre-flight cleanup for the memory-enabled agent
MEMORY_RUNTIME_NAME = "HotelBookingAgentWithMemory"

CB_PROJECT_MEMORY = f"bedrock-agentcore-{MEMORY_RUNTIME_NAME.lower()}-builder"

# No pre-flight delete of the CodeBuild project here.
#
# The previous version did a blind codebuild.delete_project(name=CB_PROJECT_MEMORY)
# wrapped in a bare except: pass. That is a name-constructed deletion that bypasses
# the WorkshopResource tag gate every other teardown path in this workshop enforces
# (08-cleanup/workshop_cleanup.py deletes only tagged resources). It is also
# unnecessary: the starter toolkit's create_or_update_project catches
# ResourceAlreadyExistsException on create_project and falls back to update_project,
# so launch() below is idempotent on re-runs without any manual delete. If a stale
# CodeBuild project ever must be removed, run the tag-gated teardown in
# 08-cleanup/workshop_cleanup.py rather than deleting by name here. See bug V7.

# Remove only the config file THIS notebook's toolkit run writes, in THIS directory.
# The previous version globbed ~/.bedrock_agentcore*.yaml. HOME is shared with every
# other AgentCore project on the machine, so running this workshop destroyed unrelated
# local config. Exact path, current directory only. See bug B45.
local_cfg = os.path.join(os.getcwd(), ".bedrock_agentcore.yaml")
if os.path.exists(local_cfg):
    os.remove(local_cfg)
    print(f"Deleted stale config: {local_cfg}")

print("✅ Pre-flight cleanup done")

# Configure and deploy the memory-enabled agent
agent_runtime_memory = Runtime()

agent_runtime_memory.configure(
    entrypoint="booking_agent_with_memory.py",
    execution_role=AGENTCORE_ROLE_ARN,
    auto_create_ecr=True,
    requirements_file="agent_requirements.txt",
    region=REGION,
    agent_name=MEMORY_RUNTIME_NAME,
    # NO_MEMORY here does NOT disable memory for this agent.
    #
    # It stops the STARTER TOOLKIT creating a memory of its own. In non-interactive
    # mode memory_mode="STM_AND_LTM" makes the toolkit unconditionally create a second
    # memory named "HotelBookingAgentWithMemory_memory". That copy is untagged, so
    # Module 8 cannot see it, and it survives teardown as a silent billable leak.
    # See bug B40.
    #
    # This agent does its own memory wiring: booking_agent_with_memory.py builds an
    # AgentCoreMemoryConfig / AgentCoreMemorySessionManager from the
    # BEDROCK_AGENTCORE_MEMORY_ID env var passed in launch() below. That points it at
    # the tagged workshop memory created in Step 1. STM and LTM both still work; the
    # toolkit's duplicate was never used by anything.
    memory_mode="NO_MEMORY",
    deployment_type="container",
    non_interactive=True,
)

print("\nLaunching agent with long-term memory (3-5 minutes)...")

result_memory = agent_runtime_memory.launch(
    auto_update_on_conflict=True,
    env_vars={
        "AWS_REGION": REGION,
        "BOOKINGS_TABLE": BOOKINGS_TABLE,
        "GATEWAY_URL": GATEWAY_URL,
        "BEDROCK_AGENTCORE_MEMORY_ID": MEMORY_ID,  # ← Pass memory ID to agent
    },
)

MEMORY_RUNTIME_ARN = result_memory.agent_arn
MEMORY_RUNTIME_ID = MEMORY_RUNTIME_ARN.split("/")[-1] if MEMORY_RUNTIME_ARN else None

print(f"\n✅ Memory-enabled agent deployed!")
print(f"Runtime ARN: {MEMORY_RUNTIME_ARN}")
print(f"Memory ID: {MEMORY_ID}")
print("\nThis agent will remember conversations across different session IDs.")

---

### Tag the toolkit-created resources

The ECR repository, CodeBuild project and Runtime for the memory-enabled agent are all
created by the starter toolkit without the workshop tag. Module 8 deletes only tagged
resources, so tag them now or teardown will leave them running.

In [ ]:
# --- Tag the resources the starter toolkit created for the memory agent ---
# Same reasoning as Module 6: the toolkit creates the ECR repo, the CodeBuild project
# and the Runtime, and does not forward tags. Module 8 deletes only tagged resources.
#
# Every target is addressed by EXACT name or by ARN. The toolkit's shared
# AmazonBedrockAgentCoreSDKCodeBuild-* IAM role is deliberately NOT tagged, because
# tagging it would make it eligible for deletion. See bug B6.

ecr_client = boto3.client("ecr", region_name=REGION)

ECR_REPO_MEMORY = f"bedrock-agentcore-{MEMORY_RUNTIME_NAME.lower()}"

try:
    repo = ecr_client.describe_repositories(repositoryNames=[ECR_REPO_MEMORY])["repositories"][0]
    ecr_client.tag_resource(resourceArn=repo["repositoryArn"], tags=WORKSHOP_TAGS_KV)
    print(f"Tagged ECR repository: {ECR_REPO_MEMORY}")
except ecr_client.exceptions.RepositoryNotFoundException:
    print(f"ECR repository not found (nothing to tag): {ECR_REPO_MEMORY}")

# update_project REPLACES the whole tag set, so merge rather than clobber.
projects = codebuild_client.batch_get_projects(names=[CB_PROJECT_MEMORY])["projects"]
if projects:
    merged = [t for t in projects[0].get("tags", []) if t.get("key") != WORKSHOP_TAG_KEY]
    codebuild_client.update_project(name=CB_PROJECT_MEMORY, tags=merged + WORKSHOP_TAGS_KV_LOWER)
    print(f"Tagged CodeBuild project: {CB_PROJECT_MEMORY}")
else:
    print(f"CodeBuild project not found (nothing to tag): {CB_PROJECT_MEMORY}")

if not MEMORY_RUNTIME_ARN:
    raise RuntimeError("MEMORY_RUNTIME_ARN is not set. Re-run the launch cell before tagging.")
agentcore.tag_resource(resourceArn=MEMORY_RUNTIME_ARN, tags=WORKSHOP_TAGS_MAP)
print(f"Tagged AgentCore Runtime: {MEMORY_RUNTIME_ARN}")

runtime_tags = agentcore.list_tags_for_resource(resourceArn=MEMORY_RUNTIME_ARN).get("tags", {})
if runtime_tags.get(WORKSHOP_TAG_KEY) != WORKSHOP_TAG_VALUE:
    raise RuntimeError(f"Runtime tag did not stick. Read back: {runtime_tags}")
print("\nAll toolkit-created resources tagged and verified.")

---
## Step 3: Test Long-Term Memory

Now test that the agent remembers information **across different sessions**.

**Test scenario:**
1. **Session A:** Tell the agent your name and preferences
2. **Wait for records:** Poll AgentCore until both fact and preference records have been extracted
3. **Session B:** Ask about hotels (different session ID, same actor ID)
4. **Verify:** The agent remembers your name and preferences from Session A

**Actor ID** is how AgentCore identifies whose memory to use:
- Same actor ID across sessions → Agent recalls strategies
- Different actor ID → Different memory space
- Format: `user-{8-char-uuid}` (e.g., `user-a1b2c3d4`)

In [ ]:
import json
import uuid
def invoke_agent_memory_boto3(agent_arn, prompt, session_id, user_id, region=REGION, show_tools=True):
    """
    Invoke agent with long-term memory using boto3 directly.
    
    Uses boto3 event system to add custom actor ID header.
    This is the ONLY way to properly set actor ID for LTM.
    """
    client = boto3.client('bedrock-agentcore', region_name=region)
    event_system = client.meta.events
    
    EVENT_NAME = 'before-sign.bedrock-agentcore.InvokeAgentRuntime'
    CUSTOM_HEADER_NAME = 'X-Amzn-Bedrock-AgentCore-Runtime-Custom-Actor-Id'
    
    def add_custom_runtime_header(request, **kwargs):
        request.headers.add_header(CUSTOM_HEADER_NAME, user_id)
    
    try:
        handler = event_system.register_first(EVENT_NAME, add_custom_runtime_header)
        
        payload = json.dumps({"prompt": prompt}).encode()
        response = client.invoke_agent_runtime(
            agentRuntimeArn=agent_arn,
            runtimeSessionId=session_id,
            payload=payload,
            qualifier="DEFAULT"
        )
        
        # Read StreamingBody correctly
        response_body = response['response'].read()
        content = response_body.decode('utf-8')
        
        # Try to parse as JSON (new agent with tools_used)
        try:
            result = json.loads(content)
            if isinstance(result, dict):
                response_text = result.get('response', str(result))
                tools_called = result.get('tools_used', [])
            else:
                response_text = content
                tools_called = []
        except json.JSONDecodeError:
            response_text = content
            tools_called = []
        
        # Display tools if any were used
        if show_tools and tools_called:
            print(f"\n🔧 Tools called:")
            for i, tool in enumerate(tools_called, 1):
                print(f"   {i}. {tool}")
            print()

        if not response_text or not str(response_text).strip():
            raise RuntimeError(
                f"Agent returned an empty response for session {session_id}. "
                f"Raw body: {content!r}"
            )

        return response_text

    except Exception as e:
        # Do NOT return None here. The previous version swallowed every failure and
        # returned None, so five HTTP 500s printed as "Agent: None" and the notebook
        # carried on to print success. Fail loudly instead. See bug B41.
        import traceback
        traceback.print_exc()
        raise RuntimeError(
            f"invoke_agent_runtime FAILED for session {session_id}: {type(e).__name__}: {e}"
        ) from e

    finally:
        try:
            event_system.unregister(EVENT_NAME, handler)
        except Exception:
            pass

In [ ]:
# Session A: Store preferences and facts
session_a = str(uuid.uuid4())
user_alex = f"user-{str(uuid.uuid4())[:8]}"  # 8-char user ID

# The facts we teach the agent in this session and assert on later.
FACT_NAME = "Alex"
FACT_LOYALTY = "HOTEL-12345"

print("=" * 70)
print("SESSION A: Teaching the agent about user preferences")
print("=" * 70)
print(f"\nUser ID:    {user_alex}")
print(f"Session ID: {session_a}")
print()

# Test 1: Tell the agent your name and preferences
prompt_a = f"My name is {FACT_NAME} and I prefer 4-star hotels in Paris. I also have a loyalty membership number: {FACT_LOYALTY}."
print(f"User: {prompt_a}")
print("-" * 70)

# invoke_agent_memory_boto3 raises on failure, so a broken call stops the notebook
# here rather than printing a checkmark over an HTTP 500. See bug B41.
response_a = invoke_agent_memory_boto3(
    agent_arn=MEMORY_RUNTIME_ARN,
    prompt=prompt_a,
    session_id=session_a,
    user_id=user_alex,
    region=REGION
)

print(f"Agent: {response_a}")
print()

# Test 2: Within the same session, verify STM works
prompt_a2 = "What's my loyalty number?"
print(f"User: {prompt_a2}")
print("-" * 70)

response_a2 = invoke_agent_memory_boto3(
    agent_arn=MEMORY_RUNTIME_ARN,
    prompt=prompt_a2,
    session_id=session_a,  # Same session
    user_id=user_alex,
    region=REGION
)

print(f"Agent: {response_a2}")
print()

# --- STM assertion. Conditional on the actual response, not printed regardless. ---
stm_recalled = FACT_LOYALTY in str(response_a2)
if stm_recalled:
    print(f"PASS  STM: agent recalled {FACT_LOYALTY} within Session A.")
else:
    raise AssertionError(
        f"FAIL  STM: the agent did not recall {FACT_LOYALTY!r} within the same session.\n"
        f"  prompt:   {prompt_a2}\n"
        f"  response: {response_a2}"
    )
print()

# Wait for actual extracted records instead of guessing a fixed delay.
print("Now waiting for AgentCore to extract long-term strategies...")
print("   Memory extraction is an ASYNCHRONOUS background process.")
print("   AgentCore analyzes the conversation and identifies:")
print("   - Facts: name, loyalty number")
print("   - Preferences: 4-star hotels, Paris")
print()
from memory_check import wait_for_memory_records

wait_for_memory_records(
    memory_id=MEMORY_ID,
    actor_id=user_alex,
    region=REGION,
    timeout=300,
)


In [ ]:
# Session B: NEW session ID, same user — test cross-session memory recall
session_b = str(uuid.uuid4())

print("=" * 70)
print("SESSION B: Testing cross-session memory recall")
print("=" * 70)
print(f"\nUser ID:    {user_alex}  <- SAME user as Session A")
print(f"Session ID: {session_b}  <- DIFFERENT session")
print()
print("The agent should recall:")
print(f"  - Fact: Your name is {FACT_NAME}")
print(f"  - Fact: Loyalty number is {FACT_LOYALTY}")
print("  - Preference: You prefer 4-star hotels in Paris")
print()

# Every check below records a real boolean. Nothing prints a checkmark
# unconditionally: that is the defect this demo is meant to teach against.
checks = []

# Test 1: Ask the agent what it remembers
prompt_b1 = "Do you remember me? What's my name?"
print(f"User: {prompt_b1}")
print("-" * 70)
response_b1 = invoke_agent_memory_boto3(
    agent_arn=MEMORY_RUNTIME_ARN,
    prompt=prompt_b1,
    session_id=session_b,  # Different session
    user_id=user_alex,      # Same user
    region=REGION
)
print(f"Agent: {response_b1}")
checks.append((
    f"LTM fact: name {FACT_NAME!r} recalled in a new session",
    FACT_NAME.lower() in str(response_b1).lower(),
    response_b1,
))
print()

# Test 2: Ask about loyalty number (fact)
prompt_b2 = "What's my loyalty number?"
print(f"User: {prompt_b2}")
print("-" * 70)
response_b2 = invoke_agent_memory_boto3(
    agent_arn=MEMORY_RUNTIME_ARN,
    prompt=prompt_b2,
    session_id=session_b,
    user_id=user_alex,
    region=REGION
)
print(f"Agent: {response_b2}")
checks.append((
    f"LTM fact: loyalty number {FACT_LOYALTY} recalled in a new session",
    FACT_LOYALTY in str(response_b2),
    response_b2,
))
print()

# Test 3: Ask for hotel recommendation (preference)
prompt_b3 = "Find me a hotel based on my preferences"
print(f"User: {prompt_b3}")
print("-" * 70)
response_b3 = invoke_agent_memory_boto3(
    agent_arn=MEMORY_RUNTIME_ARN,
    prompt=prompt_b3,
    session_id=session_b,
    user_id=user_alex,
    region=REGION
)
print(f"Agent: {response_b3}")
# Preference recall is graded on the stored preference surfacing without being
# restated: Paris, or the 4-star rating.
_b3 = str(response_b3).lower()
checks.append((
    "LTM preference: Paris or 4-star surfaced without being restated",
    ("paris" in _b3) or ("4-star" in _b3) or ("4 star" in _b3),
    response_b3,
))
print()

print("=" * 70)
print("MEMORY TEST RESULTS")
print("=" * 70)
print(f"{'PASS' if stm_recalled else 'FAIL'}  STM (Short-term): agent remembered within Session A")
for label, ok, _ in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {label}")

failed = [(label, resp) for label, ok, resp in checks if not ok]
passed = len(checks) - len(failed)
print()
print(f"Score: {passed + int(stm_recalled)}/{len(checks) + 1} checks passed")

if failed or not stm_recalled:
    print()
    for label, resp in failed:
        print(f"  FAILED: {label}\n    response: {resp}")
    raise AssertionError(
        f"Memory demo FAILED: {len(failed)} of {len(checks)} cross-session checks did not pass"
        f"{' and STM failed' if not stm_recalled else ''}. "
        "See the per-check output above."
    )

print()
print("Key insight:")
print("  - Same session_id -> Agent uses STM (conversation buffer)")
print("  - Different session_id + same actor_id -> Agent uses LTM (extracted strategies)")
